# publications.csv から「oare_id → 候補PDF/ページ」を引く（FTS全文検索）

目的:
- `train.csv` の各 `oare_id` について、`publications.csv`（PDF名×ページ単位OCRテキスト）を全文検索し、候補 `pdf_name/page` を上位3件出す。
- **取り違え/途中欠落/数値表記ゆれ**の修正作業で「原典の当たり」を付けるための探索支援。

注意:
- 初回は `publications.csv`（216k行）から SQLite FTS5 インデックスを作るため時間がかかります（DBは `data/index/` に作成、gitには入れない想定）。
- 検索精度はOCR品質・表記揺れ・原PDFに文字列が含まれるかに依存します。**最終確定は人間**で行ってください。

関連スクリプト:
- `scripts/publications_candidate_search.py`


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

# このノートブックは notebooks/EDA/ 配下にある想定
REPO_ROOT = Path.cwd().resolve().parents[1]
SCRIPTS_DIR = REPO_ROOT / "scripts"
sys.path.insert(0, str(SCRIPTS_DIR))

import pandas as pd

import publications_candidate_search as pcs

TRAIN_CSV = REPO_ROOT / "data/kaggle/deep-past-initiative-machine-translation/train.csv"
PUBLISHED_TEXTS_CSV = REPO_ROOT / "data/kaggle/deep-past-initiative-machine-translation/published_texts.csv"
PUBLICATIONS_CSV = REPO_ROOT / "data/kaggle/deep-past-initiative-machine-translation/publications.csv"
DB_PATH = REPO_ROOT / "data/index/publications_fts.sqlite"
CONFIRMED_PATH = REPO_ROOT / "data/index/confirmed_oare_to_publication.csv"

TRAIN_CSV, PUBLISHED_TEXTS_CSV, PUBLICATIONS_CSV

(PosixPath('/Users/koshidatatsuo/python/kaggle/Deep_Past_Challenge/data/kaggle/deep-past-initiative-machine-translation/train.csv'),
 PosixPath('/Users/koshidatatsuo/python/kaggle/Deep_Past_Challenge/data/kaggle/deep-past-initiative-machine-translation/published_texts.csv'),
 PosixPath('/Users/koshidatatsuo/python/kaggle/Deep_Past_Challenge/data/kaggle/deep-past-initiative-machine-translation/publications.csv'))

In [2]:
# 1) FTSインデックス作成（初回のみ重い）
#   - 既にDBがあり publications.csv が同一ならスキップされます

pcs.ensure_publications_fts(
    publications_csv=PUBLICATIONS_CSV,
    db_path=DB_PATH,
    chunk_size=5000,
    rebuild=False,
)

DB_PATH

PosixPath('/Users/koshidatatsuo/python/kaggle/Deep_Past_Challenge/data/index/publications_fts.sqlite')

In [3]:
# 2) 対象IDの一覧（trainの1561件）

train = pd.read_csv(TRAIN_CSV, usecols=["oare_id", "transliteration", "translation"])
pt = pd.read_csv(PUBLISHED_TEXTS_CSV, usecols=["oare_id", "label", "cdli_id", "online transcript"])
merged = train.merge(pt, on="oare_id", how="left")

print("train", train.shape, "published_texts", pt.shape, "merged", merged.shape)
merged.head(2)

train (1561, 3) published_texts (7953, 4) merged (1561, 6)


,oare_id,transliteration,translation,online transcript,cdli_id,label
0,004a7dbd-57ce-46f8-9691-409be61c676e,KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-{d}IM KIŠI...,"Seal of Mannum-balum-Aššur son of Ṣilli-Adad, ...",https://oare.byu.edu/epigraphies/004a7dbd-57ce...,P535157,Cuneiform Tablet Kt 94/k 1122 (AKT 6c 705)
1,0064939c-59b9-4448-a63d-34612af0a1b5,1 TÚG ša qá-tim i-tur₄-DINGIR il₅-qé,Itūr-ilī has received one textile of ordinary ...,https://oare.byu.edu/epigraphies/0064939c-59b9...,P535772,Cuneiform Tablet Kt 94/k 1067 (AKT 6e 1160)


In [4]:
def show_candidates(oare_id: str, topk: int = 3, max_terms: int = 12):
    row = merged.loc[merged["oare_id"] == oare_id].iloc[0]
    print("oare_id:", row["oare_id"])
    print("label:", row.get("label"))
    print("cdli_id:", row.get("cdli_id"))
    print("online transcript:", row.get("online transcript"))
    print("-" * 80)
    print("translation:")
    print(str(row.get("translation", ""))[:800])
    print("-" * 80)

    df = pcs.generate_candidates(
        train_csv=TRAIN_CSV,
        published_texts_csv=PUBLISHED_TEXTS_CSV,
        db_path=DB_PATH,
        oare_id=oare_id,
        topk=topk,
        max_terms=max_terms,
    )
    display(df[["rank", "pdf_name", "page", "score", "query", "snippet"]])
    return df


# 例: まずは1件試す
example_oare_id = merged["oare_id"].iloc[0]
df_example = show_candidates(example_oare_id, topk=3)
df_example.head(3)

oare_id: 004a7dbd-57ce-46f8-9691-409be61c676e
label: Cuneiform Tablet Kt 94/k 1122 (AKT 6c 705)
cdli_id: P535157
online transcript: https://oare.byu.edu/epigraphies/004a7dbd-57ce-46f8-9691-409be61c676e
--------------------------------------------------------------------------------
translation:
Seal of Mannum-balum-Aššur son of Ṣilli-Adad, seal of Šu-Illil son of Mannum-kī-Aššur, seal of Puzur-Aššur son of Ataya. Puzur-Aššur son of Ataya owes 22 shekels of good silver to Ali-ahum. Reckoned from the week of Ilī-dan, month of Ša-kēnātim, in the eponymy of Enna-Suen, he will pay in 14 weeks. If he has not paid in time, he will add interest at the rate 1.5 shekel per mina per month.
--------------------------------------------------------------------------------


,rank,pdf_name,page,score,query,snippet
0,1,AKT 6e.pdf,387,-26.706717,P535157 OR Cuneiform OR Tablet OR 1122 OR AKT ...,…[Illil] father of [Puzur]-Anna\nSu-[Illil] fa...
1,2,AKT 6e.pdf,372,-25.748763,P535157 OR Cuneiform OR Tablet OR 1122 OR AKT ...,"…2, [seal]; 35: 14\n482: 220;\n115: 6\n119: 27..."
2,3,AKT 6c.pdf,280,-23.703689,P535157 OR Cuneiform OR Tablet OR 1122 OR AKT ...,…kt 94/k [1122]\nSeal of [Mannum]-balum-Assur ...


,oare_id,rank,pdf_name,page,score,has_akkadian,label,cdli_id,query,snippet
0,004a7dbd-57ce-46f8-9691-409be61c676e,1,AKT 6e.pdf,387,-26.706717,0,Cuneiform Tablet Kt 94/k 1122 (AKT 6c 705),P535157,P535157 OR Cuneiform OR Tablet OR 1122 OR AKT ...,…[Illil] father of [Puzur]-Anna\nSu-[Illil] fa...
1,004a7dbd-57ce-46f8-9691-409be61c676e,2,AKT 6e.pdf,372,-25.748763,0,Cuneiform Tablet Kt 94/k 1122 (AKT 6c 705),P535157,P535157 OR Cuneiform OR Tablet OR 1122 OR AKT ...,"…2, [seal]; 35: 14\n482: 220;\n115: 6\n119: 27..."
2,004a7dbd-57ce-46f8-9691-409be61c676e,3,AKT 6c.pdf,280,-23.703689,0,Cuneiform Tablet Kt 94/k 1122 (AKT 6c 705),P535157,P535157 OR Cuneiform OR Tablet OR 1122 OR AKT ...,…kt 94/k [1122]\nSeal of [Mannum]-balum-Assur ...


In [30]:
OARE_ID = "6b00f8c2-a745-48d2-ad50-3888f34ae4ef"

In [32]:
df_example = show_candidates(OARE_ID, topk=5)
df_example.head(5)

oare_id: 6b00f8c2-a745-48d2-ad50-3888f34ae4ef
label: Cuneiform Tablet Kt 94/k 1516 (AKT 6e 1113)
cdli_id: P535729
online transcript: https://oare.byu.edu/epigraphies/6b00f8c2-a745-48d2-ad50-3888f34ae4ef
--------------------------------------------------------------------------------
translation:
Peruwa owes 0.3333 mina 5 shekels of silver, the price of a slave-girl.
--------------------------------------------------------------------------------


,rank,pdf_name,page,score,query,snippet
0,1,Keibi51-Band 51 (1990 - 1991).pdf,89,-20.424573,P535729 OR Cuneiform OR Tablet OR 1516 OR AKT ...,…ISBN [0]-943872-54-[5]. i-87 p.\n1487. Sigris...
1,2,AKT 6e.pdf,249,-19.498946,P535729 OR Cuneiform OR Tablet OR 1516 OR AKT ...,…kt 94/k [1516]\n5\nPeruwa [owes] 1/3 [mina] [...
2,3,"AKT 5, 2008.pdf",32,-18.759667,P535729 OR Cuneiform OR Tablet OR 1516 OR AKT ...,…Text 43:[5] calls him an agent\n(tamk£arum) o...
3,4,AKT 5.pdf,32,-18.759667,P535729 OR Cuneiform OR Tablet OR 1516 OR AKT ...,…Text 43:[5] calls him an agent\n(tamk£arum) o...
4,5,"Veenhof, Klaas R. - The Archive of Kuliya, Son...",32,-18.759667,P535729 OR Cuneiform OR Tablet OR 1516 OR AKT ...,…Text 43:[5] calls him an agent\n(tamk£arum) o...


,oare_id,rank,pdf_name,page,score,has_akkadian,label,cdli_id,query,snippet
0,6b00f8c2-a745-48d2-ad50-3888f34ae4ef,1,Keibi51-Band 51 (1990 - 1991).pdf,89,-20.424573,0,Cuneiform Tablet Kt 94/k 1516 (AKT 6e 1113),P535729,P535729 OR Cuneiform OR Tablet OR 1516 OR AKT ...,…ISBN [0]-943872-54-[5]. i-87 p.\n1487. Sigris...
1,6b00f8c2-a745-48d2-ad50-3888f34ae4ef,2,AKT 6e.pdf,249,-19.498946,0,Cuneiform Tablet Kt 94/k 1516 (AKT 6e 1113),P535729,P535729 OR Cuneiform OR Tablet OR 1516 OR AKT ...,…kt 94/k [1516]\n5\nPeruwa [owes] 1/3 [mina] [...
2,6b00f8c2-a745-48d2-ad50-3888f34ae4ef,3,"AKT 5, 2008.pdf",32,-18.759667,0,Cuneiform Tablet Kt 94/k 1516 (AKT 6e 1113),P535729,P535729 OR Cuneiform OR Tablet OR 1516 OR AKT ...,…Text 43:[5] calls him an agent\n(tamk£arum) o...
3,6b00f8c2-a745-48d2-ad50-3888f34ae4ef,4,AKT 5.pdf,32,-18.759667,0,Cuneiform Tablet Kt 94/k 1516 (AKT 6e 1113),P535729,P535729 OR Cuneiform OR Tablet OR 1516 OR AKT ...,…Text 43:[5] calls him an agent\n(tamk£arum) o...
4,6b00f8c2-a745-48d2-ad50-3888f34ae4ef,5,"Veenhof, Klaas R. - The Archive of Kuliya, Son...",32,-18.759667,0,Cuneiform Tablet Kt 94/k 1516 (AKT 6e 1113),P535729,P535729 OR Cuneiform OR Tablet OR 1516 OR AKT ...,…Text 43:[5] calls him an agent\n(tamk£arum) o...


↑どのPDFファイルの何ページ目に該当のoare_idがありそうか確かめる

In [12]:
publications = pd.read_csv(PUBLICATIONS_CSV)

In [36]:
PDF_NAME = "AKT 6e.pdf"
PAGE = 249

In [37]:
publications[(publications["pdf_name"] == PDF_NAME) & (publications["page"] == PAGE)]

,pdf_name,page,page_text,has_akkadian
58347,AKT 6e.pdf,249,break\na-na na-ap-fa-lar ku-pa-im\ni-sa-qit-lu...,False
62405,AKT 6e.pdf,249,break\na-na na-ap-fa-lar ku-pa-im\ni-sa-qit-lu...,False


以下のpprintの表示結果をPDFと見比べる

In [38]:
from IPython.display import display
import pprint

pprint.pprint(
    publications[(publications["pdf_name"] == PDF_NAME) & (publications["page"] == PAGE)].iloc[0]["page_text"]
    )

('break\\na-na na-ap-fa-lar ku-pa-im\\ni-sa-qit-lu\\na.:.ha-ma 4 1/2 GIN '
 'KB\\ni-na Ii-bi\\nWa-al-di-lim '
 'i-n[a]\\ne-ra-bi-su-ma\\nu-se-ba-lam\\nbreak\\nSit-ka-li-a\\nNotes:\\nKULTEPE '
 "TABLETLERi VI-e\\n235\\n1112. kt 94/k 1514\\nrev.\\n5'\\nLe.\\nthey are to "
 'pay at the melting of the snow.\\nFurther, 4 ½ shekels of silver is owed '
 "by\\nWaldi-ilum.\\nHe will send it when he arrives.\\nSukkalliya.\\n1. l ': "
 'for the word kuppa \'um with the meaning "snow, ice", see comments to no. '
 '329 in volume\\n2. Note that we have the form ina na-ap-fi-ir : ku-nu-ti in '
 'line 23 there, referring to people being\\nreleased from the snow and ice '
 'that is blocking their travel.\\nComment:\\nA fragment of a note concerning '
 'outstanding debts.\\n1/3 ma-na r 51 GIN\\nKB si-im\\nam-tim '
 'Pe-ru-a\\n{haJ-bu-ul\\na-na fha-ar 1-pe\\ni-[sa-qal]\\nComment:\\n1113. kt '
 '94/k 1516\\n5\\nPeruwa owes 1/3 mina 5 shekels of silver,\\nthe price of a '
 'slave-girl.\\nHe will pay by har

あとは<gap>や加工ルールなどを自分で入れていく。  
長文に関してはCodexに頼んだ方が良いかも。

In [7]:
# # 3) 確定した対応（oare_id → pdf_name/page）をローカルCSVに追記していく

# from datetime import datetime


# def append_confirmation(oare_id: str, pdf_name: str, page: int, note: str = ""):
#     CONFIRMED_PATH.parent.mkdir(parents=True, exist_ok=True)
#     row = {
#         "oare_id": oare_id,
#         "pdf_name": pdf_name,
#         "page": int(page),
#         "note": note,
#         "confirmed_at": datetime.now().strftime("%Y-%m-%d %H:%M"),
#     }
#     if CONFIRMED_PATH.exists():
#         df = pd.read_csv(CONFIRMED_PATH)
#         df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
#     else:
#         df = pd.DataFrame([row])
#     df.to_csv(CONFIRMED_PATH, index=False)
#     return row


# CONFIRMED_PATH

In [8]:
# # 4) ipywidgets が使える場合の簡易UI（なければこのセルはスキップでOK）

# try:
#     import ipywidgets as widgets
#     from IPython.display import clear_output

#     oare_dropdown = widgets.Dropdown(
#         options=merged["oare_id"].tolist(),
#         description="oare_id",
#         layout=widgets.Layout(width="80%"),
#     )
#     topk_slider = widgets.IntSlider(value=3, min=1, max=10, step=1, description="topk")
#     search_btn = widgets.Button(description="検索", button_style="primary")
#     out = widgets.Output()

#     candidate_choice = widgets.Dropdown(description="確定", options=[("(未選択)", None)])
#     note_text = widgets.Text(description="メモ", layout=widgets.Layout(width="80%"))
#     confirm_btn = widgets.Button(description="保存", button_style="success")

#     state = {"last_df": None}

#     def on_search(_):
#         with out:
#             clear_output(wait=True)
#             df = show_candidates(oare_dropdown.value, topk=int(topk_slider.value))
#             state["last_df"] = df
#             opts = [("(未選択)", None)]
#             for _, r in df.dropna(subset=["rank"]).iterrows():
#                 key = f"#{int(r['rank'])}: {r['pdf_name']} p{int(r['page'])}"
#                 opts.append((key, (r["pdf_name"], int(r["page"]))))
#             candidate_choice.options = opts

#     def on_confirm(_):
#         choice = candidate_choice.value
#         if choice is None:
#             with out:
#                 print("[WARN] 候補が未選択です")
#             return
#         pdf_name, page = choice
#         row = append_confirmation(oare_dropdown.value, pdf_name, page, note=note_text.value)
#         with out:
#             print("[OK] saved:", row)

#     search_btn.on_click(on_search)
#     confirm_btn.on_click(on_confirm)

#     display(widgets.VBox([widgets.HBox([oare_dropdown, topk_slider, search_btn]), out]))
#     display(widgets.HBox([candidate_choice, note_text, confirm_btn]))

# except Exception as e:
#     print("ipywidgets が使えませんでした:", repr(e))
#     print("上の show_candidates() と append_confirmation() を手動で呼び出してください")